In [1]:
# Find the fraction of area that has no template coverage

In [2]:
import numpy as np
import healpy as hp
import matplotlib.pylab as plt
%matplotlib inline
import rubin_sim.splat as splat
import copy 

import sqlite3
import pandas as pd
from os.path import basename

from rubin_scheduler.scheduler.utils import get_current_footprint

import glob
import pandas as pd

In [3]:
# Define a report element that we want to run. Needs branch maf-proto of rubin_sim to run
class FracCoverageElement(splat.BaseReportElement):
    """Estimate how many alerts in certain years one would get
    """
    
    def compute_elements(self,visits_df=None, visits_array=None, nside=32, bands="ugrizy",
                         seeing_limit=1.2, n_for_template=6, years=np.arange(10)+1):

        current_footprint, labels = get_current_footprint(nside)
        
        times = np.arange(3653)
        metric = splat.AccumulateCountMetric(times)
        
        is_too = np.array(["ToO" in note for note in visits_array["scheduler_note"]])
        is_ddf = np.array(["DD" in note for note in visits_array["scheduler_note"]])

        count_arrays = {}
        good_seeing_count = {}
        count_subbed = {}
        area_no_template = years*0.

        pix_area = hp.nside2pixarea(nside, degrees=True)

        tot_area = 0
        for band in bands:
            area = np.where(current_footprint[band] > 0)[0].size * pix_area
            tot_area += area
        
        for band in bands:
            indx = np.where((~is_too) & (~is_ddf) & (visits_array["visitExposureTime"] > 20) & (visits_array["band"] == band))
            sub_data = visits_array[indx]
            sl = splat.Slicer(nside=nside, missing=0)
            count_arrays[band] = sl(sub_data, metric)

            indx = np.where((~is_too) & (~is_ddf) & (visits_array["visitExposureTime"] > 20) &
                            (visits_array["band"] == band) &
                            (visits_array["seeingFwhm500"] < seeing_limit))
            sub_data = visits_array[indx]
            sl = splat.Slicer(nside=nside, missing=0)
            good_seeing_count[band] = sl(sub_data, metric)
        
            # Need to find the spot where good_seeing_count >= n_for_template
            mask = np.ones(good_seeing_count[band].shape) 
            mask[np.where(good_seeing_count[band] >= 6)] = 0
            count_masked = count_arrays[band] * mask
            count_subbed[band] = count_arrays[band] - np.max(count_masked, axis=1)[:,np.newaxis]
            count_subbed[band][np.where(count_subbed[band] < 0)] = 0

            for i,year in enumerate(years):
                indx_missing = np.where((current_footprint[band] > 0) & (count_subbed[band][:, int(365.25 * (year))] == 0))[0]
                area_no_template[i] += indx_missing.size * pix_area
                
        frac_no_area = area_no_template/tot_area    
        return area_no_template, frac_no_area, None


In [4]:
db_files = ['baseline_v5.3.7_10yrs.db', 
            'template_var_1_v5.3.7_10yrs.db',
            'template_var_2_v5.3.7_10yrs.db',
            'template_var_3_v5.3.7_10yrs.db',
            'three_stripes_v5.3.7_10yrs.db',
            'three_stripes_kr_v5.3.7_10yrs.db',
            'desi_tskr_v5.3.7_10yrs.db',
            'desi_stripes_delayed_v5.3.7_10yrs.db',
            'desi_stripes_v5.3.7_10yrs.db',
            "double_stripes_timing_bands_v5.3.7_10yrs.db",
            "three_stripes_or_v5.3.7_10yrs.db",
            "delay_roll_v5.3.7_10yrs.db"
            ]

In [5]:
areas = {}
frac_no_templates = {}

for filename in db_files:
    print(filename)
    element = FracCoverageElement(observations=filename)
    area_no_template, frac_no_area, _tmp2, = element()
    areas[filename.replace(".db", "")] = area_no_template
    frac_no_templates[filename.replace(".db", "")] = frac_no_area
    


baseline_v5.3.7_10yrs.db
template_var_1_v5.3.7_10yrs.db
template_var_2_v5.3.7_10yrs.db
template_var_3_v5.3.7_10yrs.db
three_stripes_v5.3.7_10yrs.db
three_stripes_kr_v5.3.7_10yrs.db
desi_tskr_v5.3.7_10yrs.db
desi_stripes_delayed_v5.3.7_10yrs.db
desi_stripes_v5.3.7_10yrs.db
double_stripes_timing_bands_v5.3.7_10yrs.db
three_stripes_or_v5.3.7_10yrs.db
delay_roll_v5.3.7_10yrs.db


In [6]:
areas

{'baseline_v5.3.7_10yrs': array([1.78769546e+04, 6.72442069e+03, 2.95767081e+03, 8.39293645e+02,
        6.41220345e+02, 2.14859173e+02, 6.04291425e+01, 3.02145712e+01,
        2.68573966e+01, 1.67858729e+01]),
 'template_var_1_v5.3.7_10yrs': array([4.24850443e+04, 1.25894047e+03, 6.11005774e+02, 1.37644158e+02,
        1.20858285e+02, 6.71434916e+01, 3.69289204e+01, 3.69289204e+01,
        2.35002221e+01, 2.35002221e+01]),
 'template_var_2_v5.3.7_10yrs': array([3.12082949e+04, 1.83301732e+03, 9.40008883e+02, 1.57787205e+02,
        7.72150154e+01, 4.02860950e+01, 3.02145712e+01, 3.02145712e+01,
        3.02145712e+01, 3.02145712e+01]),
 'template_var_3_v5.3.7_10yrs': array([1.75345228e+04, 1.12801066e+03, 5.43862282e+02, 1.41001332e+02,
        8.39293645e+01, 3.69289204e+01, 2.01430475e+01, 1.34286983e+01,
        1.00715237e+01, 6.71434916e+00]),
 'three_stripes_v5.3.7_10yrs': array([17873.59746842,  6798.27852621,  2252.66414375,   587.50555165,
          402.8609497 ,   211.501998

In [7]:
frac_no_templates

{'baseline_v5.3.7_10yrs': array([1.13986643e-01, 4.28761024e-02, 1.88586352e-02, 5.35148557e-03,
        4.08853498e-03, 1.36998031e-03, 3.85306961e-04, 1.92653481e-04,
        1.71247538e-04, 1.07029711e-04]),
 'template_var_1_v5.3.7_10yrs': array([2.70892200e-01, 8.02722836e-03, 3.89588150e-03, 8.77643634e-04,
        7.70613922e-04, 4.28118846e-04, 2.35465365e-04, 2.35465365e-04,
        1.49841596e-04, 1.49841596e-04]),
 'template_var_2_v5.3.7_10yrs': array([1.98989640e-01, 1.16876445e-02, 5.99366384e-03, 1.00607929e-03,
        4.92336673e-04, 2.56871307e-04, 1.92653481e-04, 1.92653481e-04,
        1.92653481e-04, 1.92653481e-04]),
 'template_var_3_v5.3.7_10yrs': array([1.11803237e-01, 7.19239661e-03, 3.46776265e-03, 8.99049576e-04,
        5.35148557e-04, 2.35465365e-04, 1.28435654e-04, 8.56237692e-05,
        6.42178269e-05, 4.28118846e-05]),
 'three_stripes_v5.3.7_10yrs': array([0.11396524, 0.04334703, 0.01436339, 0.00374604, 0.00256871,
        0.00134857, 0.00029968, 0.000214

In [8]:
nice = pd.DataFrame.from_dict(frac_no_templates).T
cn = np.arange(10)
cnn = cn+1
nice.columns=cnn
print("Fraction of footprint with no template image per year")
nice

Fraction of footprint with no template image per year


,1,2,3,4,5,6,7,8,9,10
baseline_v5.3.7_10yrs,0.113987,0.042876,0.018859,0.005351,0.004089,0.001370,0.000385,0.000193,0.000171,0.000107
template_var_1_v5.3.7_10yrs,0.270892,0.008027,0.003896,0.000878,0.000771,0.000428,0.000235,0.000235,0.000150,0.000150
template_var_2_v5.3.7_10yrs,0.198990,0.011688,0.005994,0.001006,0.000492,0.000257,0.000193,0.000193,0.000193,0.000193
template_var_3_v5.3.7_10yrs,0.111803,0.007192,0.003468,0.000899,0.000535,0.000235,0.000128,0.000086,0.000064,0.000043
three_stripes_v5.3.7_10yrs,0.113965,0.043347,0.014363,0.003746,0.002569,0.001349,0.000300,0.000214,0.000171,0.000150
three_stripes_kr_v5.3.7_10yrs,0.426513,0.225554,0.129549,0.034785,0.016119,0.005459,0.001991,0.001220,0.000728,0.000492
desi_tskr_v5.3.7_10yrs,0.426578,0.221188,0.126937,0.035106,0.016675,0.008220,0.004174,0.002290,0.001605,0.001584
desi_stripes_delayed_v5.3.7_10yrs,0.113965,0.039430,0.016697,0.006079,0.004367,0.001862,0.000642,0.000514,0.000300,0.000278
desi_stripes_v5.3.7_10yrs,0.367069,0.156349,0.069184,0.023718,0.015348,0.007513,0.002954,0.002098,0.001242,0.000985
double_stripes_timing_bands_v5.3.7_10yrs,0.430987,0.243749,0.135928,0.041592,0.020464,0.008413,0.004324,0.002612,0.000813,0.000728
